### Querying notebook
Connect to the same database:

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
#Check how many documents are in the index:
sqlite_index.count()

79

In [4]:
#Try a search:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

### RAG with sqlitesearch
We use the RAGBase class from rag_helper.py with this sqlitesearch index.
Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [8]:
#
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, so we just connect to the database file.

In [9]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after the course has started. If you want a certificate, make sure to submit your project while submissions are still open.


In [10]:
#closing the db
sqlite_index.close()